<a href="https://colab.research.google.com/github/januvishwanath56-debug/Gen-AI-Assignments/blob/main/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment – Task 5: Fine-Tuning BERT for POS Tagging & Chunking
**Data Science Internship – February 2026**

---

## Objective
Build and fine-tune a transformer model (DistilBERT) to perform:
- **Part-of-Speech (POS) Tagging** – Grammar-level tagging
- **Chunking (Phrase Detection)** – Phrase-level grouping

## Pipeline
`Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison`


---
## Step 0: Install Required Libraries

In [ ]:
# Install required packages
!pip install transformers datasets seqeval evaluate accelerate -q

---
## Step 1: Import Libraries

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Data Handling ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Hugging Face ──────────────────────────────────────────────────────────────
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch

print("✅ All libraries imported successfully!")
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

---
## Task 1: Dataset Selection (10%)

We use the **CoNLL-2003** dataset, which provides:
- `pos_tags` – Part-of-Speech labels (grammar-level)
- `chunk_tags` – Chunking / Phrase labels (phrase-level)

This single dataset lets us train **two separate models** and compare them directly.

In [ ]:
# ── Load the CoNLL-2003 dataset ───────────────────────────────────────────────
# The dataset has: tokens, pos_tags, chunk_tags, ner_tags
# Note: trust_remote_code is no longer supported in newer datasets versions
# dataset = load_dataset("conll2003")
# load_dataset("eriktks/conll2003")
# Primary: load the pure-Parquet branch (no .py script)
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")

# Fallback if that fails:
dataset = load_dataset("BramVanroy/conll2003")

print("Dataset structure:")
print(dataset)
print("\nSample record:")
print(dataset["train"][0])

In [ ]:
# ── Inspect label sets ────────────────────────────────────────────────────────

# POS tag labels
pos_feature   = dataset["train"].features["pos_tags"]
POS_LABELS    = pos_feature.feature.names

# Chunk tag labels
chunk_feature = dataset["train"].features["chunk_tags"]
CHUNK_LABELS  = chunk_feature.feature.names

print(f"POS Labels   ({len(POS_LABELS)})  : {POS_LABELS}")
print()
print(f"Chunk Labels ({len(CHUNK_LABELS)}) : {CHUNK_LABELS}")

### Label Summary

| Task | Dataset | Label Type | # Labels |
|------|---------|-----------|----------|
| POS Tagging | CoNLL-2003 | Penn Treebank POS tags (NN, VB, JJ…) | 47 |
| Chunking    | CoNLL-2003 | IOB2 Chunk tags (B-NP, I-NP, B-VP…) | 23 |

---
## Task 2: Data Preprocessing (15%)

Key challenges with BERT tokenization:
1. **Subword tokenization** – a single word may be split into multiple tokens (e.g., `"playing"` → `["play", "##ing"]`)
2. **Special tokens** – `[CLS]` and `[SEP]` have no corresponding labels
3. **Label alignment** – only the *first* subword of each word gets its real label; the rest are set to `-100` so PyTorch ignores them in the loss

In [ ]:
# ── Model checkpoint ─────────────────────────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"

# Load the tokenizer (fast tokenizer gives word_ids())
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

In [ ]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenize a batch of sentences and align the token-level labels.

    Strategy:
    - Use `is_split_into_words=True` because the input is already word-tokenized.
    - For each word, the FIRST subword keeps the original label.
    - Subsequent subwords and special tokens receive label = -100
      (ignored by PyTorch's CrossEntropyLoss).

    Args:
        examples     : HuggingFace dataset batch (dict of lists)
        label_column : 'pos_tags' or 'chunk_tags'

    Returns:
        tokenized batch with aligned 'labels' field added
    """
    # Tokenise all sentences in the batch at once (fast tokenizer)
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True   # crucial – input is list-of-words
    )

    all_labels = []

    for i, word_labels in enumerate(examples[label_column]):
        # word_ids() maps each token position to its word index (None for specials)
        word_ids      = tokenized_inputs.word_ids(batch_index=i)
        label_ids     = []
        previous_word = None

        for word_id in word_ids:
            if word_id is None:
                # Special tokens [CLS] / [SEP] → ignore
                label_ids.append(-100)
            elif word_id != previous_word:
                # First subword of a new word → use the real label
                label_ids.append(word_labels[word_id])
            else:
                # Subsequent subwords → ignore in loss
                label_ids.append(-100)

            previous_word = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


print("✅ tokenize_and_align_labels() defined")

In [ ]:
# ── Apply preprocessing to the full dataset (both tasks) ─────────────────────

# POS Tagging dataset
pos_tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, "pos_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Chunking dataset
chunk_tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, "chunk_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("POS tokenized split sizes  :", {k: len(v) for k, v in pos_tokenized.items()})
print("Chunk tokenized split sizes:", {k: len(v) for k, v in chunk_tokenized.items()})

In [ ]:
# ── Verify alignment on one example ──────────────────────────────────────────
sample_idx = 0
sample     = dataset["train"][sample_idx]
tokens_out = tokenizer(sample["tokens"], is_split_into_words=True)

print("Original words :", sample["tokens"][:8])
print("Original POS   :", [POS_LABELS[t] for t in sample["pos_tags"][:8]])
print()
print("Subword tokens :", tokenizer.convert_ids_to_tokens(tokens_out["input_ids"][:12]))
print("Word IDs       :", tokens_out.word_ids()[:12])

---
## Task 3: Model Setup (15%)

We use `AutoModelForTokenClassification`, which adds a **linear classification head** on top of DistilBERT. Two separate models are instantiated – one for each task.

In [ ]:
# ── Helper: build label mappings ──────────────────────────────────────────────
def build_label_maps(label_list):
    """
    Returns (id2label, label2id) dicts from a list of label strings.
    These are required by AutoModelForTokenClassification.
    """
    id2label  = {i: lbl for i, lbl in enumerate(label_list)}
    label2id  = {lbl: i for i, lbl in enumerate(label_list)}
    return id2label, label2id


# ── POS Tagging Model ─────────────────────────────────────────────────────────
pos_id2label, pos_label2id = build_label_maps(POS_LABELS)

pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels = len(POS_LABELS),
    id2label   = pos_id2label,
    label2id   = pos_label2id,
    ignore_mismatched_sizes=True
)

print(f"POS Model  – num_labels : {pos_model.config.num_labels}")
print(f"id2label sample         : { {0: pos_id2label[0], 1: pos_id2label[1]} }")

In [ ]:
# ── Chunking Model ────────────────────────────────────────────────────────────
chunk_id2label, chunk_label2id = build_label_maps(CHUNK_LABELS)

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels = len(CHUNK_LABELS),
    id2label   = chunk_id2label,
    label2id   = chunk_label2id,
    ignore_mismatched_sizes=True
)

print(f"Chunk Model – num_labels : {chunk_model.config.num_labels}")
print(f"id2label sample          : { {0: chunk_id2label[0], 1: chunk_id2label[1]} }")

---
## Task 4: Training (20%)

We use the Hugging Face `Trainer` API with the following hyperparameters:

| Parameter | Value |
|-----------|-------|
| Learning Rate | 2e-5 |
| Epochs | 3 |
| Batch Size (train) | 16 |
| Batch Size (eval) | 16 |
| Weight Decay | 0.01 |

In [ ]:
# ── Shared: seqeval metric loader ─────────────────────────────────────────────
seqeval = evaluate.load("seqeval")

# ── Data collator (pads inputs + labels to same length in a batch) ────────────
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("✅ Metric and data collator ready")

In [ ]:
def make_compute_metrics(label_list):
    """
    Returns a compute_metrics function bound to a specific label list.
    Used by the Trainer to evaluate after each epoch.

    The function:
    1. Converts logits → predicted class IDs (argmax)
    2. Removes padded / special-token positions (label == -100)
    3. Maps IDs back to string labels
    4. Computes precision, recall, F1 via seqeval
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)  # shape (batch, seq_len)

        true_labels = [
            [label_list[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision" : results["overall_precision"],
            "recall"    : results["overall_recall"],
            "f1"        : results["overall_f1"],
            "accuracy"  : results["overall_accuracy"],
        }

    return compute_metrics


print("✅ compute_metrics factory defined")

In [ ]:
# ── Training Arguments (shared hyperparameters) ───────────────────────────────
COMMON_ARGS = dict(
    num_train_epochs              = 3,
    learning_rate                 = 2e-5,
    per_device_train_batch_size   = 16,
    per_device_eval_batch_size    = 16,
    weight_decay                  = 0.01,
    eval_strategy                 = "epoch",   # evaluate after every epoch
    save_strategy                 = "epoch",
    load_best_model_at_end        = True,
    push_to_hub                   = False,
    logging_steps                 = 50,
    report_to                     = "none",    # disable wandb / mlflow
)

print("✅ Shared training arguments defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║          TRAIN MODEL 1: POS TAGGING                      ║
# ╚══════════════════════════════════════════════════════════╝

pos_args = TrainingArguments(
    output_dir="./pos_model",
    **COMMON_ARGS
)

pos_trainer = Trainer(
    model           = pos_model,
    args            = pos_args,
    train_dataset   = pos_tokenized["train"],
    eval_dataset    = pos_tokenized["validation"],

    processing_class = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(POS_LABELS),
)

print("🚀 Starting POS Tagging training…")
pos_train_result = pos_trainer.train()
print("✅ POS Tagging training complete!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║          TRAIN MODEL 2: CHUNKING                         ║
# ╚══════════════════════════════════════════════════════════╝

chunk_args = TrainingArguments(
    output_dir="./chunk_model",
    **COMMON_ARGS
)

chunk_trainer = Trainer(
    model           = chunk_model,
    args            = chunk_args,
    train_dataset   = chunk_tokenized["train"],
    eval_dataset    = chunk_tokenized["validation"],
     processing_class = tokenizer,
    # tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(CHUNK_LABELS),
)

print("🚀 Starting Chunking training…")
chunk_train_result = chunk_trainer.train()
print("✅ Chunking training complete!")

---
## Task 5: Evaluation (15%)

We evaluate both models on the **test split** using the `seqeval` metric, which computes entity-level Precision, Recall, and F1.

In [ ]:
# ── Evaluate POS Model on test set ───────────────────────────────────────────
pos_eval = pos_trainer.evaluate(eval_dataset=pos_tokenized["test"])

print("="*50)
print(" POS Tagging – Test Set Evaluation")
print("="*50)
print(f"  Precision : {pos_eval['eval_precision']:.4f}")
print(f"  Recall    : {pos_eval['eval_recall']:.4f}")
print(f"  F1 Score  : {pos_eval['eval_f1']:.4f}")
print(f"  Accuracy  : {pos_eval['eval_accuracy']:.4f}")

In [ ]:
# ── Evaluate Chunking Model on test set ───────────────────────────────────────
chunk_eval = chunk_trainer.evaluate(eval_dataset=chunk_tokenized["test"])

print("="*50)
print(" Chunking – Test Set Evaluation")
print("="*50)
print(f"  Precision : {chunk_eval['eval_precision']:.4f}")
print(f"  Recall    : {chunk_eval['eval_recall']:.4f}")
print(f"  F1 Score  : {chunk_eval['eval_f1']:.4f}")
print(f"  Accuracy  : {chunk_eval['eval_accuracy']:.4f}")

In [ ]:
# ── Side-by-side comparison table ─────────────────────────────────────────────
import pandas as pd

eval_df = pd.DataFrame({
    "Metric"    : ["Precision", "Recall", "F1 Score", "Accuracy"],
    "POS Tagging" : [
        round(pos_eval['eval_precision'], 4),
        round(pos_eval['eval_recall'],    4),
        round(pos_eval['eval_f1'],        4),
        round(pos_eval['eval_accuracy'],  4),
    ],
    "Chunking" : [
        round(chunk_eval['eval_precision'], 4),
        round(chunk_eval['eval_recall'],    4),
        round(chunk_eval['eval_f1'],        4),
        round(chunk_eval['eval_accuracy'],  4),
    ],
})

eval_df = eval_df.set_index("Metric")
print(eval_df.to_string())

---
## Task 6: Inference (10%)

Run both fine-tuned models on custom sentences to see real predictions.

In [ ]:
# ── Build inference pipelines ─────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1

pos_pipeline = pipeline(
    "token-classification",
    model     = pos_model,
    tokenizer = tokenizer,
    aggregation_strategy = "simple",  # merges subword predictions
    device    = device
)

chunk_pipeline = pipeline(
    "token-classification",
    model     = chunk_model,
    tokenizer = tokenizer,
    aggregation_strategy = "simple",
    device    = device
)

print("✅ Inference pipelines ready")

In [ ]:
# ── Run inference ─────────────────────────────────────────────────────────────
test_sentences = [
    "John works at Google in California.",
    "The quick brown fox jumps over the lazy dog.",
    "She is reading a fascinating book about machine learning.",
]

for sentence in test_sentences:
    print("\n" + "─"*60)
    print(f"📝 Input: {sentence}")

    # ── POS Tags ──────────────────────────────────────────────
    pos_preds = pos_pipeline(sentence)
    print("\n📌 POS Tags:")
    print(f"  {'Word':<20} {'POS Tag':<12} {'Score':>6}")
    print(f"  {'────':<20} {'───────':<12} {'─────':>6}")
    for p in pos_preds:
        print(f"  {p['word']:<20} {p['entity_group']:<12} {p['score']:>6.3f}")

    # ── Chunk Tags ────────────────────────────────────────────
    chunk_preds = chunk_pipeline(sentence)
    print("\n🧩 Chunk Tags:")
    print(f"  {'Word':<20} {'Chunk Tag':<12} {'Score':>6}")
    print(f"  {'────':<20} {'─────────':<12} {'─────':>6}")
    for p in chunk_preds:
        print(f"  {p['word']:<20} {p['entity_group']:<12} {p['score']:>6.3f}")

---
## Task 7: Comparison – POS Tagging vs Chunking (10%)

### Detailed Comparison Table

In [ ]:
comparison_data = {
    "Aspect"             : [
        "Granularity",
        "Output Level",
        "Label Set Size",
        "Task Difficulty",
        "Label Type",
        "Example Input",
        "Example Output",
        "Use Case",
        "Ambiguity",
        "Typical F1 (BERT)"
    ],
    "POS Tagging"        : [
        "Word-level",
        "Grammar tag per token",
        "~47 (Penn Treebank)",
        "Easier (local context)",
        "Flat (NN, VB, JJ…)",
        "John / works / at / Google",
        "NNP / VBZ / IN / NNP",
        "Grammar analysis, parsing",
        "Low ('bank' as noun or verb)",
        "~97–98%"
    ],
    "Chunking"           : [
        "Phrase-level",
        "Phrase group per token",
        "~23 (IOB2)",
        "Medium (span boundaries)",
        "IOB2 (B-NP, I-NP, B-VP…)",
        "John / works / at / Google",
        "B-NP / B-VP / B-PP / B-NP",
        "Info extraction, QA, NER",
        "Medium (phrase boundaries)",
        "~94–96%"
    ],
}

comp_df = pd.DataFrame(comparison_data).set_index("Aspect")
print(comp_df.to_string())

In [ ]:
# ── Visual: metric comparison bar chart ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

metrics = ["Precision", "Recall", "F1 Score", "Accuracy"]
pos_scores   = [
    pos_eval['eval_precision'],
    pos_eval['eval_recall'],
    pos_eval['eval_f1'],
    pos_eval['eval_accuracy']
]
chunk_scores = [
    chunk_eval['eval_precision'],
    chunk_eval['eval_recall'],
    chunk_eval['eval_f1'],
    chunk_eval['eval_accuracy']
]

x    = np.arange(len(metrics))
w    = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, pos_scores,   w, label='POS Tagging', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x + w/2, chunk_scores, w, label='Chunking',    color='#DD8452', alpha=0.85)

# Annotate bars
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('POS Tagging vs Chunking – Evaluation Metrics (Test Set)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("comparison_chart.png", bbox_inches='tight')
plt.show()
print("✅ Chart saved as comparison_chart.png")

---
## Task 8: Report / Blog (5%)

### Understanding POS Tagging vs Chunking

---

#### 1. What is POS Tagging?

**Part-of-Speech (POS) Tagging** assigns a grammatical category to every word in a sentence. For example:

| Word | POS Tag | Meaning |
|------|---------|--------|
| John | NNP | Proper noun |
| works | VBZ | Verb, 3rd person singular |
| at | IN | Preposition |
| Google | NNP | Proper noun |

POS tagging is a **word-level**, flat labeling task. It is a foundational NLP step used in parsing, named entity recognition, and text-to-speech.

---

#### 2. What is Chunking?

**Chunking (Shallow Parsing)** groups consecutive words into **meaningful phrases** using IOB2 notation:
- `B-XP` = Beginning of an X-phrase
- `I-XP` = Inside the same X-phrase
- `O` = Outside any phrase

For the same sentence:

| Word | Chunk Tag | Phrase |
|------|-----------|-------|
| John | B-NP | Noun Phrase |
| works | B-VP | Verb Phrase |
| at | B-PP | Prepositional Phrase |
| Google | B-NP | Noun Phrase |

Chunking is a **phrase-level** task and is used in information extraction, question answering, and relation detection.

---

#### 3. Key Differences

| | POS Tagging | Chunking |
|---|---|---|
| Granularity | Word | Phrase |
| Output | Flat tag per token | IOB2 span labels |
| Difficulty | Easier | Slightly harder |
| F1 Score (BERT) | ~97–98% | ~94–96% |
| Ambiguity source | Lexical (e.g. "bank") | Phrase boundary |

---

#### 4. Challenges Faced

1. **Subword Tokenization Alignment** – BERT splits words into subwords (e.g., `"playing"` → `["play", "##ing"]`). Only the first subword inherits its label; the rest are masked with `-100`. Getting this alignment correct is the most error-prone step.

2. **Special Token Handling** – `[CLS]` and `[SEP]` tokens must be masked with `-100` so they do not contribute to the loss.

3. **Class Imbalance** – In chunking, the `O` (outside) tag is very frequent, which can skew metrics. Seqeval computes entity-level F1, which mitigates this.

4. **Compute Time** – Training two large transformer models sequentially is time-consuming without a GPU. We used DistilBERT (40% smaller than BERT-base) to reduce this.

5. **IOB2 Consistency** – Predictions can sometimes produce invalid IOB2 sequences (e.g., `I-NP` without a preceding `B-NP`). The `aggregation_strategy="simple"` in the pipeline helps merge these correctly.

---

#### 5. Observations & Insights

- **DistilBERT performs very well** on both tasks despite being a smaller, distilled model. This demonstrates that pre-trained contextual representations transfer effectively to token classification.
- **POS tagging consistently achieves higher F1** than chunking, confirming that grammatical tagging (local context) is an easier problem than identifying phrase boundaries (global span structure).
- **Label alignment is the most critical preprocessing step.** Any mismatch here silently corrupts training without raising an error.
- The **Hugging Face Trainer API** dramatically reduces boilerplate code while still giving full control over hyperparameters and evaluation.
- **seqeval** is the correct metric for both tasks as it computes entity/span-level precision, recall, and F1, which is more informative than raw token accuracy.

---

#### 6. Summary

This assignment demonstrated the full token classification pipeline:
- Loading and exploring a structured NLP benchmark (CoNLL-2003)
- Handling BERT's subword tokenization and label alignment
- Fine-tuning DistilBERT for both POS tagging (47 labels) and chunking (23 IOB2 labels)
- Evaluating models with seqeval and comparing results side-by-side
- Running zero-shot inference on unseen sentences with both models

Both tasks are excellent demonstrations of how pre-trained transformers can be adapted to sequence labeling problems with minimal task-specific architecture changes.

---
## Save Models (Optional)

In [ ]:
# Save fine-tuned models and tokenizer to disk for later use
pos_model.save_pretrained("./saved_pos_model")
chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_tokenizer")

print("✅ Models and tokenizer saved successfully!")
print("   ./saved_pos_model/")
print("   ./saved_chunk_model/")
print("   ./saved_tokenizer/")